### archive

In [2]:
import pandas as pd

# Load and clean both sheets
file_path = "Data for technical assessment.xlsx"
sheet1_raw = pd.read_excel(file_path, sheet_name="Dataset 1 - General", engine="openpyxl", header=None)
sheet2_raw = pd.read_excel(file_path, sheet_name="Dataset 2 - Underwriting", engine="openpyxl", header=None)

# Merge first two rows into header
header1 = [f"{a} {b}".strip() for a, b in zip(sheet1_raw.iloc[0], sheet1_raw.iloc[1])]
sheet1 = sheet1_raw.iloc[2:].copy()
sheet1.columns = header1
sheet1.rename(columns={sheet1.columns[0]: "Firm"}, inplace=True)

header2 = [f"{a} {b}".strip() for a, b in zip(sheet2_raw.iloc[0], sheet2_raw.iloc[1])]
sheet2 = sheet2_raw.iloc[2:].copy()
sheet2.columns = header2
sheet2.rename(columns={sheet2.columns[0]: "Firm"}, inplace=True)

# Merge sheets
merged = pd.merge(sheet1, sheet2, on="Firm", how="inner")

In [3]:
merged

,Firm,NWP (£m) 2016YE,NWP (£m) 2017YE,NWP (£m) 2018YE,NWP (£m) 2019YE,NWP (£m) 2020YE,SCR (£m) 2016YE,SCR (£m) 2017YE,SCR (£m) 2018YE,SCR (£m) 2019YE,...,Gross expense ratio 2016YE,Gross expense ratio 2017YE,Gross expense ratio 2018YE,Gross expense ratio 2019YE,Gross expense ratio 2020YE,Gross combined ratio 2016YE,Gross combined ratio 2017YE,Gross combined ratio 2018YE,Gross combined ratio 2019YE,Gross combined ratio 2020YE
0,Firm 1,-13779.815629,0,0,0,0,1085.360139,0.0,0,0,...,0,56.813725,0,0,0,0,68.215239,0,0,0
1,Firm 2,28.178059,26.865049,25.064438,23.226445,21.718558,10.190314,10.113572,9.495235,8.146471,...,0.743265,0.963451,0.814588,0,0,0.945394,1.126744,0.939197,0,0
2,Firm 3,0,75.609681,70.578732,78.432782,85.73583,322.955115,363.782327,362.290859,394.295982,...,0,0,0,0,0,0,0,0,0,0
3,Firm 4,22344.199923,23963.910709,25760.390158,25512.748836,24996.021042,16573.6448,16332.7488,17103.616,17219.24608,...,0.14393,0.147519,0.092971,0.054781,-0.546237,0.848032,1.474778,1.727968,1.208823,-10.736084
4,Firm 5,68.200993,51.663132,44.010833,42.008556,81.273653,52.824396,38.053768,34.696815,57.231788,...,0.177212,0.13431,0.109074,0.121044,0.109187,0.508711,1.259454,1.304168,0.983277,0.997184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,Firm 321,0,0,-1.011367,-6.599067,24.632234,0,0.258621,62.227588,51.830942,...,0.211938,0.256118,0.245704,0.236224,0.278674,0.978004,1.002691,0.97254,0.958443,0.81687
321,Firm 322,2092.156137,2084.124818,2022.212247,2103.048716,2029.697013,1711.220667,1641.309461,1329.471064,1399.098954,...,0.364543,0.372169,0.39877,0.420327,0.373813,0.885956,0.960993,0.913687,0.943246,0.995833
322,Firm 323,0,0,0,0,0,30.438558,15.232621,5.332069,1.55137,...,0,0,0,0,0,0,0,0,0,0
323,Firm 324,23.41538,22.650321,24.268465,25.811984,26.546638,32.096633,30.205948,29.517977,29.954935,...,0.427635,0.371681,0.357627,0.330893,0.302577,1.063136,1.006945,0.982816,0.994712,0.780065


In [28]:
# Extract key metrics
metrics = ["GWP (£m)", "NWP (£m)", "SCR coverage ratio", "Gross claims incurred (£m)", "Net combined ratio"]
metric_cols = [col for col in merged.columns if any(m in col for m in metrics)]

# Tidy dataframe
records = []
for _, row in merged.iterrows():
    firm = row['Firm']
    for col in metric_cols:
        parts = col.split()
        metric = " ".join(parts[:-1])
        year = parts[-1]
        try:
            value = float(row[col])
        except:
            value = None
        records.append({"Firm": firm, "Year": year, "Metric": metric, "Value": value})

tidy_df = pd.DataFrame(records)

In [20]:
#some data sense checks
# for each year, check if NWP <= GWP for each firm


### start

We go into this with the assumption we care most about the latest data, i.e. 2020, but we also look at previous years in case we need to sense check. 2019 data is still useful for calculating year-on-year differences.
what we want:
1. 4x4 plot - largest firms (avg across all years), most volatile (biggest YoY change in a chosen metric) 2019 - 2020 (done), low SCR coverage (2020) and NCR vs NWP for 2020.
1. we potentially have a list of firms using this, we can write that down in the markldown aloing wuith the reasong for each
something like

* largest - 1,222,2,...

* scr - ,...

* volatile - ....

* Outliers - from the 4 plots, which are issues vs outliers that are extremely far? 

*Gives us suggested methodology:*

1. suggest a combined metric using some kind of ranking, so 
`x*size + y*scr + z*volatile = score`

1. rank each firm on all metrics, and then do a weighted rank for every firm across these.


In [14]:
import pandas as pd
tidy_df_updated = pd.read_csv('data/cleaned_results/combined_long.csv')

In [28]:
# YoY change table
# Ensure correct ordering
tidy_df_updated = tidy_df_updated.sort_values(by=['Firm', 'Metric', 'Year'])

# Calculate YoY change by firm and metric
tidy_df_updated['YoY_change'] = tidy_df_updated.groupby(['Firm', 'Metric'])['Value'].pct_change()

In [15]:
# Summary for prioritization
summary = tidy_df_updated.groupby(['Firm', 'Metric']).agg({"Value": ["mean", "std"]}).reset_index()
summary.columns = ['Firm', 'Metric', 'AvgValue', 'StdDev']

# Identify top firms and outliers
largest_firms = summary[summary['Metric'] == 'GWP (£m)'].sort_values('AvgValue', ascending=False).head(10)
volatile_firms = summary.sort_values('StdDev', ascending=False).head(10) # pick which metric we care about here
outliers_scr = summary[(summary['Metric'] == 'SCR coverage ratio') & (summary['AvgValue'] < 1)].sort_values('AvgValue').head(10)
outliers_combined = summary[(summary['Metric'] == 'Net combined ratio') & (summary['AvgValue'] > 1)].sort_values('AvgValue', ascending=False).head(10)

In [16]:
summary

,Firm,Metric,AvgValue,StdDev
0,Firm 1,EoF for SCR (£m),446.978924,996.784143
1,Firm 1,Excess of assets over liabilities (£m) [= equity],407.170771,907.770621
2,Firm 1,GWP (£m),281.896959,630.340763
3,Firm 1,"Gross BEL (inc. TPs as whole, pre-TMTP) (£m)",1.534894,3.432127
4,Firm 1,Gross claims incurred (£m),0.009335,0.020873
...,...,...,...,...
3837,Firm 99,Pure net claims ratio,-7470.127880,16703.713740
3838,Firm 99,SCR (£m),257.514827,14.817735
3839,Firm 99,SCR coverage ratio,1.491485,0.124202
3840,Firm 99,Total assets (£m),1044.942180,139.034947


In [17]:
# # Detect reporting errors: extreme year-on-year changes (>300% change)
# error_records = []
# for firm in tidy_df_updated['Firm'].unique():
#     firm_data = tidy_df[tidy_df['Firm'] == firm]
#     for metric in metrics:
#         metric_data = firm_data[firm_data['Metric'] == metric].sort_values('Year')
#         values = metric_data['Value'].tolist()
#         for i in range(1, len(values)):
#             if values[i-1] and values[i] and abs(values[i] - values[i-1]) > 3 * abs(values[i-1]):
#                 error_records.append({"Firm": firm, "Metric": metric, "PrevValue": values[i-1], "CurrentValue": values[i]})
# errors_df = pd.DataFrame(error_records)

In [18]:
# errors_df['year-on-year change'] = ((errors_df['CurrentValue'] - errors_df['PrevValue']) / errors_df['PrevValue']).abs() * 100

# #print the top 20 errors detected
# errors_df.sort_values('year-on-year change', ascending=False).head(20)

In [19]:
import plotly.express as px

# Create charts
fig_gwp = px.bar(largest_firms, x='Firm', y='AvgValue', title='Top 10 Firms by GWP (£m)')
# fig_gwp.write_image('largest_firms.png')
# fig_gwp.write_json('largest_firms.json')

fig_vol = px.bar(volatile_firms, x='Firm', y='StdDev', title='Top 10 Most Volatile Firms')
# fig_vol.write_image('volatile_firms.png')
# fig_vol.write_json('volatile_firms.json')

fig_scr = px.bar(outliers_scr, x='Firm', y='AvgValue', title='Low SCR Coverage Ratio (<1)')
# fig_scr.write_image('low_scr.png')
# fig_scr.write_json('low_scr.json')

fig_combined = px.bar(outliers_combined, x='Firm', y='AvgValue', title='High Net Combined Ratio (>1)')
# fig_combined.write_image('high_combined.png')
# fig_combined.write_json('high_combined.json')


In [20]:
fig_gwp.show()
fig_vol.show()
fig_scr.show()
fig_combined.show()

In [21]:
df_combined = tidy_df_updated[tidy_df_updated["Metric"] == "Net combined ratio"]
df_nwp = tidy_df_updated[tidy_df_updated["Metric"] == "NWP (£m)"]
df_gwp = tidy_df_updated[tidy_df_updated["Metric"] == "GWP (£m)"]

df_plot = pd.merge(df_combined, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
df_plot = pd.merge(df_plot, df_gwp, on=["Firm", "Year"], suffixes=("", "_gwp"))

fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)


In [22]:
fig_combined_vs_nwp.show()

In [23]:
# plotting the same as above, restricting net combined ratios up to 1000 and >= 0

df_combined_restricted = tidy_df_updated[
    (tidy_df_updated["Metric"] == "Net combined ratio") &
    (tidy_df_updated["Value"] <= 1000) &
    (tidy_df_updated["Value"] >= -1000)
]

df_nwp = tidy_df_updated[tidy_df_updated["Metric"] == "NWP (£m)"]

df_plot = pd.merge(df_combined_restricted, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))

fig_combined_vs_nwp_restricted = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium - outliers removed",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)

In [24]:
fig_combined_vs_nwp_restricted.show()

In [25]:
#plot NWP vs GWP for 2020
fig_gwp_vs_nwp = px.scatter(
    df_plot,#[df_plot["Year"] == '2020YE'],
    x="Value",
    y="Value_nwp",
    title="Gross Written Premium vs Net Written Premium for 2020",
    labels={"Value": "Gross Written Premium (£m)", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm"]
)


ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['Firm', 'Year', 'Metric_combined', 'Value_combined', 'Metric_nwp', 'Value_nwp'] but received: Value

In [ ]:
fig_gwp_vs_nwp.show()

### Use only 2020 data

In [90]:
import pandas as pd
tidy_df_updated = pd.read_csv('data/cleaned_results/combined_long.csv')

In [91]:
# YoY change table
# Ensure correct ordering
tidy_df_updated = tidy_df_updated.sort_values(by=['Firm', 'Metric', 'Year'])

# Calculate YoY change by firm and metric
tidy_df_updated['YoY_change'] = tidy_df_updated.groupby(['Firm', 'Metric'])['Value'].pct_change()

In [92]:
tidy_df_updated_2020 = tidy_df_updated[tidy_df_updated["Year"]==2020]
tidy_df_updated_2020

,Firm,Year,Metric,Value,YoY_change
14,Firm 1,2020,EoF for SCR (£m),0.000000,NaN
39,Firm 1,2020,Excess of assets over liabilities (£m) [= equity],0.000000,NaN
24,Firm 1,2020,GWP (£m),0.000000,NaN
49,Firm 1,2020,"Gross BEL (inc. TPs as whole, pre-TMTP) (£m)",0.000000,NaN
44,Firm 1,2020,Gross claims incurred (£m),0.000000,NaN
...,...,...,...,...,...
6009,Firm 99,2020,Pure net claims ratio,0.000000,NaN
5959,Firm 99,2020,SCR (£m),281.473991,0.146232
5969,Firm 99,2020,SCR coverage ratio,1.378122,-0.166187
5979,Firm 99,2020,Total assets (£m),1215.246388,0.086642


#### Visualise selected metrics (Top 10 firms)

In [72]:
# Identify top firms and outliers
largest_firms = tidy_df_updated_2020[tidy_df_updated_2020['Metric'] == 'GWP (£m)'].sort_values('Value', ascending=False).head(10)
# volatile_firms = tidy_df_updated_2020.sort_values('StdDev', ascending=False).head(10) # pick which metric we care about here
outliers_scr = tidy_df_updated_2020[(tidy_df_updated_2020['Metric'] == 'SCR coverage ratio') & (tidy_df_updated_2020['Value'] < 1)].sort_values('Value').head(10)
outliers_combined = tidy_df_updated_2020[(tidy_df_updated_2020['Metric'] == 'Net combined ratio') & (tidy_df_updated_2020['Value'] > 1)].sort_values('Value', ascending=False).head(10)

In [73]:
tidy_df_updated_2020[tidy_df_updated_2020["Metric"]=="SCR coverage ratio"]

,Firm,Year,Metric,Value,YoY_change
19,Firm 1,2020,SCR coverage ratio,0.000000,NaN
529,Firm 10,2020,SCR coverage ratio,1.421303,-0.147538
6054,Firm 100,2020,SCR coverage ratio,1.077108,0.008382
6139,Firm 102,2020,SCR coverage ratio,0.000000,NaN
6224,Firm 104,2020,SCR coverage ratio,8.080187,1.577630
...,...,...,...,...,...
5629,Firm 92,2020,SCR coverage ratio,1.946785,-0.213788
5714,Firm 94,2020,SCR coverage ratio,13.616788,0.002486
5799,Firm 96,2020,SCR coverage ratio,0.000000,NaN
5884,Firm 97,2020,SCR coverage ratio,1.475222,-0.459535


In [74]:
import plotly.express as px

# Create charts
fig_gwp = px.bar(largest_firms, x='Firm', y='Value', title='Top 10 Firms by GWP (£m) in 2020')
# fig_gwp.write_image('largest_firms.png')
# fig_gwp.write_json('largest_firms.json')

# fig_vol = px.bar(volatile_firms, x='Firm', y='StdDev', title='Top 10 Most Volatile Firms')
# fig_vol.write_image('volatile_firms.png')
# fig_vol.write_json('volatile_firms.json')

fig_scr = px.bar(outliers_scr, x='Firm', y='Value', title='Low SCR Coverage Ratio (<1) in 2020')
# fig_scr.write_image('low_scr.png')
# fig_scr.write_json('low_scr.json')

fig_combined = px.bar(outliers_combined, x='Firm', y='Value', title='High Net Combined Ratio (>1) in 2020')
# fig_combined.write_image('high_combined.png')
# fig_combined.write_json('high_combined.json')

In [76]:
fig_gwp.show()
# fig_vol.show()
fig_scr.show()
fig_combined.show()

#### Visualise NWP vs. GWP

In [87]:
df_combined = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "Net combined ratio"]
df_nwp = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "NWP (£m)"]
df_gwp = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "GWP (£m)"]

df_plot = pd.merge(df_combined, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
df_plot = pd.merge(df_plot, df_gwp, on=["Firm", "Year"], suffixes=("", "_gwp"))

fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium in 2020",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)
fig_combined_vs_nwp.show()

In [88]:
import plotly.express as px

# Create a new column for text labels, only if condition is met
# Add conditional text labels
df_plot['label'] = df_plot.apply(
    lambda row: f"{row['Firm']}" 
                if (row['Value_combined'] > 40 or row['Value_nwp'] > 20000) 
                else '',
    axis=1
)

# Create the scatter plot
fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium in 2020",
    labels={
        "Value_combined": "Net Combined Ratio",
        "Value_nwp": "Net Written Premium (£m)"
    },
    hover_data=["Firm", "Year"],
    text="label"  # Add labels
)

# Position text next to the points
fig_combined_vs_nwp.update_traces(textposition="top center")
fig_combined_vs_nwp.show()


In [89]:
# plotting the same as above, restricting net combined ratios up to 200 and >= 0

df_combined_restricted = tidy_df_updated_2020[
    (tidy_df_updated_2020["Metric"] == "Net combined ratio") &
    (tidy_df_updated_2020["Value"] <= 200) &
    (tidy_df_updated_2020["Value"] >= -200)
]

df_nwp = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "NWP (£m)"]

df_plot = pd.merge(df_combined_restricted, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
# Add conditional text labels
df_plot['label'] = df_plot.apply(
    lambda row: f"{row['Firm']}" 
                if (row['Value_combined'] > 10 or row['Value_nwp'] > 20000) 
                else '',
    axis=1
)

fig_combined_vs_nwp_restricted = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium in 2020- outliers removed",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"],
    text="label"  # Add labels
)

# Position text next to the points
fig_combined_vs_nwp_restricted.update_traces(textposition="top center")
fig_combined_vs_nwp_restricted.show()

#### Visualise YoY change for all metrics

In [93]:
# plot the most volitile (YoY change for 2020-2019)
# missing value in YoY change means both current and previour year have value 0
# -1 in YoY change means current year has value 0 while preious year doesn't
for i in tidy_df_updated_2020["Metric"].unique():
    temp_df = tidy_df_updated_2020[tidy_df_updated_2020["Metric"]==i].sort_values(by="YoY_change", ascending=False).head(10)
    fig_YoY = px.bar(temp_df, x='Firm', y='YoY_change', title=f'Top 10 Firms by 2019 - 2020 YoY change for "{i}" ')
    fig_YoY.show()